# Content Based Filtering



Recommendation systems are a collection of algorithms used to recommend items to users based on information taken from the user. These systems have become ubiquitous, and can be commonly seen in online stores, movies databases and job finders. In this notebook, we will explore Content-based recommendation systems and implement a simple version of one using Python and the Pandas library.


### Table of contents

<div class="alert alert-block alert-info" style="margin-top: 20px">
    <ol>
        <li><a href="https://#ref1">Acquiring the Data</a></li>
        <li><a href="https://#ref2">Preprocessing</a></li>
        <li><a href="https://#ref3">Content-Based Filtering</a></li>
    </ol>
</div>
<br>


<a id="ref1"></a>

# Acquiring the Data


In [1]:
import pandas as pd
import os

In [2]:
#Storing the movie information into a pandas dataframe
# movies_df = pd.read_csv('movies.csv')
#Storing the user information into a pandas dataframe
# ratings_df = pd.read_csv('ratings.csv')

# movies_df.head()

In [46]:
dataset_path = os.path.join(os.path.expanduser('~'), 'datasets', 'movies', '7')

credits_df = pd.read_csv(dataset_path + '/credits.csv')
# movies_meta_df = pd.read_csv(dataset_path + '/movies_metadata.csv')
ratings_df = pd.read_csv(dataset_path + '/ratings_small.csv')

In [47]:
credits_df

,cast,crew,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",8844
2,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...",15602
3,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de...",31357
4,"[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de...",11862
...,...,...,...
45471,"[{'cast_id': 0, 'character': '', 'credit_id': ...","[{'credit_id': '5894a97d925141426c00818c', 'de...",439050
45472,"[{'cast_id': 1002, 'character': 'Sister Angela...","[{'credit_id': '52fe4af1c3a36847f81e9b15', 'de...",111109
45473,"[{'cast_id': 6, 'character': 'Emily Shaw', 'cr...","[{'credit_id': '52fe4776c3a368484e0c8387', 'de...",67758
45474,"[{'cast_id': 2, 'character': '', 'credit_id': ...","[{'credit_id': '533bccebc3a36844cf0011a7', 'de...",227506


In [48]:
# Dropping useless bits from movies_meta_df
movies_df = pd.read_csv(f"{dataset_path}/movies_metadata.csv", usecols=['id', 'title', 'release_date', 'genres', 'popularity', 'vote_average', 'vote_count'])

/var/folders/lr/44zcpqfx2196g_qy5t3kc7m40000gp/T/ipykernel_2813/1038718327.py:2: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies_df = pd.read_csv(f"{dataset_path}/movies_metadata.csv", usecols=['id', 'title', 'release_date', 'genres', 'popularity', 'vote_average', 'vote_count'])


In [6]:
# from google.colab import drive
# drive.mount('/content/drive')

In [7]:
# ratings_small_df = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/Datasets/movie_datasets/7/ratings_small.csv')

In [8]:
# movies_meta_df = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/Datasets/movie_datasets/7/movies_metadata.csv')
# movies_df = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/Datasets/movie_datasets/movies/movies.csv')

In [9]:
# ratings_df = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/Datasets/movie_datasets/movies/ratings.csv')

In [10]:
# credits_df = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/Datasets/movie_datasets/7/credits.csv')
# credits_df.head()

In [49]:
movies_df = movies_df.sort_values(by='release_date', ascending=False)
# movies_meta_df.head(3)
movies_df = movies_df.drop(35587) # A weird film entry is now gone!
movies_df.head(3)

,genres,id,popularity,release_date,title,vote_average,vote_count
26559,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",76600,6.020055,2020-12-16,Avatar 2,0.0,58.0
38885,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",299782,0.238154,2018-12-31,The Other Side of the Wind,0.0,1.0
30402,"[{'id': 53, 'name': 'Thriller'}, {'id': 28, 'n...",38700,2.178546,2018-11-07,Bad Boys for Life,0.0,12.0


<a id="ref2"></a>

# Preprocessing


In [12]:

from math import sqrt
import numpy as np
import matplotlib.pyplot as plt

import ast
%matplotlib inline

First, let's get all of the imports out of the way:


Now let's read each file into their Dataframes:


In [13]:
# Make a genres table / format it to be able to use it as a feature


In [50]:
# prompt: print the dimensions of ratings, credits, movies, movies_metadata

print("ratings_df dimensions:", ratings_df.shape)
print("credits_df dimensions:", credits_df.shape)
print("movies_df dimensions:", movies_df.shape)
# print("movies_meta_df dimensions:", movies_meta_df.shape)


ratings_df dimensions: (100004, 4)
credits_df dimensions: (45476, 3)
movies_df dimensions: (45465, 7)


In [51]:


def get_first_3_cast(cast_series):
    """
    Extracts names and IDs of the first 3 cast members from a Pandas Series.

    Args:
        cast_series (pandas.Series): A Pandas Series containing cast information
                                      (string representation of list of dictionaries).

    Returns:
        pandas.DataFrame: DataFrame with a column 'cast_info' containing tuples of (name, ID)
                         for the first 3 cast members.
    """

    all_cast_info = []

    for cast_list_str in cast_series:
        cast_list = ast.literal_eval(cast_list_str)  # Convert string to list
        cast_info = []
        for i in range(min(3, len(cast_list))):
            cast_info.append((cast_list[i]['name'], cast_list[i]['id']))  # Create (name, ID) tuple
        all_cast_info.append(cast_info)

    return pd.DataFrame({'cast_info': all_cast_info})


# Usage:
cast_info_df = get_first_3_cast(credits_df['cast'])
cast_info_df.head()

,cast_info
0,"[(Tom Hanks, 31), (Tim Allen, 12898), (Don Ric..."
1,"[(Robin Williams, 2157), (Jonathan Hyde, 8537)..."
2,"[(Walter Matthau, 6837), (Jack Lemmon, 3151), ..."
3,"[(Whitney Houston, 8851), (Angela Bassett, 978..."
4,"[(Steve Martin, 67773), (Diane Keaton, 3092), ..."


In [52]:
def get_directors_from_crew(credits):
    """
    Extracts the names of the director(s) from the 'crew' column of a Pandas DataFrame.

    Args:
        credits (pandas.DataFrame): A Pandas DataFrame containing 'crew' column with crew information.

    Returns:
        pandas.DataFrame: DataFrame with a column 'director_name' containing the names of the director(s), and the corresponding director 'id', and 'credits id' for the film.

    """
    # Set 'id' column as index
    global director_credit_id
    credits = credits.set_index('id')

    director_info = []

    for index, row in credits.iterrows():  # Iterate using index and row
        crew_list_str = row['crew']
        crew_list = ast.literal_eval(crew_list_str)
        director_name = None
        director_id = None
        film_id = index  # Get the film 'id' (now index)

        for crew_member_l in crew_list:
            if crew_member_l['job'] == 'Director':
                director_name = crew_member_l['name']
                director_credit_id = crew_member_l['credit_id']
                break

        director_info.append((director_name, director_credit_id, film_id))

    return pd.DataFrame({'director_info': director_info})


director_info_df = get_directors_from_crew(credits_df)
director_info_df.head()


,director_info
0,"(John Lasseter, 52fe4284c3a36847f8024f49, 862)"
1,"(Joe Johnston, 52fe44bfc3a36847f80a7c7d, 8844)"
2,"(Howard Deutch, 52fe466a9251416c75077a89, 15602)"
3,"(Forest Whitaker, 52fe44779251416c91011acb, 31..."
4,"(Charles Shyer, 52fe44959251416c75039eef, 11862)"


In [53]:
def get_first_3_crew(crew_series):
    """
    Extracts names and IDs of the first 3 crew members from a Pandas Series.

    Args:
        crew_series (pandas.Series): A Pandas Series containing crew information
                                      (string representation of list of dictionaries).

    Returns:
        pandas.DataFrame: DataFrame with columns 'crew_name' and 'crew_id' for the first 3 crew members.
    """

    all_crew_info = []

    for crew_list_str in crew_series:
        crew_list_s = ast.literal_eval(crew_list_str)  # Convert string to list
        crew_info = []
        for i in range(min(3, len(crew_list_s))):
            crew_info.append((crew_list_s[i]['name'], crew_list_s[i]['job'], crew_list_s[i]['id']))  # Create (name, job, ID) tuple
        all_crew_info.append(crew_info)

    return pd.DataFrame({'crew_info': all_crew_info})


crew_info_df = get_first_3_crew(credits_df['crew'])
# crew_info_df.head()

In [54]:
# find all the unique jobs of from crew_info_df
# dataframe example: [(John Lasseter, Director, 7879), (Joss Whedon...
unique_jobs = set()
for crew_list in crew_info_df['crew_info']:
    for crew_member in crew_list:
        unique_jobs.add(crew_member[1])

In [55]:
top_3_credits_df = pd.concat([cast_info_df, director_info_df, credits_df['id']], axis=1)
# top_3_credits_df = pd.concat([cast_info_df, director_info_df, credits_df['id']], axis=1)
# top_3_credits_df.head()
top_3_credits_df.head()

,cast_info,director_info,id
0,"[(Tom Hanks, 31), (Tim Allen, 12898), (Don Ric...","(John Lasseter, 52fe4284c3a36847f8024f49, 862)",862
1,"[(Robin Williams, 2157), (Jonathan Hyde, 8537)...","(Joe Johnston, 52fe44bfc3a36847f80a7c7d, 8844)",8844
2,"[(Walter Matthau, 6837), (Jack Lemmon, 3151), ...","(Howard Deutch, 52fe466a9251416c75077a89, 15602)",15602
3,"[(Whitney Houston, 8851), (Angela Bassett, 978...","(Forest Whitaker, 52fe44779251416c91011acb, 31...",31357
4,"[(Steve Martin, 67773), (Diane Keaton, 3092), ...","(Charles Shyer, 52fe44959251416c75039eef, 11862)",11862


In [56]:
# movies_df.head()
# Make a separate table for genres with the genre name and the corresponding id. Take it from the genres column on movies_df
# Create a separate table for genres with the genre name and the corresponding id
import json

pre_genre_steal_df = movies_df.copy()

def extract_genres():
    genres_set = set()
    for genres_str in movies_df['genres']:
        genres_list = json.loads(genres_str.replace("'", "\""))
        for genre in genres_list:
            genres_set.add((genre['id'], genre['name']))
    return pd.DataFrame(genres_set, columns=['genre_id', 'genre_name'])

genres_df = extract_genres()
drop_entries = [7759, 7760, 7761, 11602, 11176, 33751, 29812, 2883]
genres_df = genres_df[~genres_df['genre_id'].isin(drop_entries)]
genres_df

,genre_id,genre_name
0,10751,Family
1,10769,Foreign
2,53,Thriller
3,16,Animation
5,36,History
6,9648,Mystery
8,80,Crime
9,99,Documentary
10,14,Fantasy
12,10402,Music


In [59]:
movies_df 

,genres,id,popularity,release_date,title,vote_average,vote_count
26559,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",76600,6.020055,2020-12-16,Avatar 2,0.0,58.0
38885,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",299782,0.238154,2018-12-31,The Other Side of the Wind,0.0,1.0
30402,"[{'id': 53, 'name': 'Thriller'}, {'id': 28, 'n...",38700,2.178546,2018-11-07,Bad Boys for Life,0.0,12.0
38130,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...",332283,3.328261,2018-04-25,Mary Shelley,0.0,1.0
44535,"[{'id': 18, 'name': 'Drama'}]",412059,0.155147,2018-04-04,Mobile Homes,0.0,1.0
...,...,...,...,...,...,...,...
45148,[],438910,0.001586,NaN,Engineering Red,6.0,2.0
45203,"[{'id': 9648, 'name': 'Mystery'}, {'id': 878, ...",433711,0.00022,NaN,All Superheroes Must Die 2: The Last Superhero,4.0,1.0
45338,[],335251,0.0,NaN,The Land Where the Blues Began,0.0,0.0
45410,"[{'id': 18, 'name': 'Drama'}, {'id': 80, 'name...",449131,0.008903,NaN,Aprel,6.0,1.0


In [65]:
ohe_movies_df = movies_df.copy()
ohe_movies_df.head(3)

,genres,id,popularity,release_date,title,vote_average,vote_count
26559,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",76600,6.020055,2020-12-16,Avatar 2,0.0,58.0
38885,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",299782,0.238154,2018-12-31,The Other Side of the Wind,0.0,1.0
30402,"[{'id': 53, 'name': 'Thriller'}, {'id': 28, 'n...",38700,2.178546,2018-11-07,Bad Boys for Life,0.0,12.0


In [68]:
sampleGenre = ohe_movies_df['genres']
type(sampleGenre.iloc[0])

str

In [69]:



# Bad genres: (7759,GoHands), (7760,BROSTA TV), (7761,Mardock Scramble Production Committee), (11602,Vision View Entertainment), (11176,Carousel Productions), (33751,Sentai Filmworks), (29812,Telescene Film Group Productions)
drop_entries = [7759, 7760, 7761, 11602, 11176, 33751, 29812, 2883]
# Drop the genre entries with id's listed above
ohe_movies_df['genres'] = ohe_movies_df['genres'].apply(
    lambda x: json.loads(x.replace("'", "\"")) if isinstance(x, str) else x
).apply(
    lambda x: [genre for genre in x if genre['id'] not in drop_entries] if isinstance(x, list) else []
)

# List all the genres (validation check - there's no dodgy genres left in)
all_genres = set()
for genres in ohe_movies_df['genres']:
    for genre in genres:
        all_genres.add(genre['name'])
print(all_genres)

{'Science Fiction', 'Adventure', 'Drama', 'Music', 'Animation', 'Horror', 'War', 'Western', 'Fantasy', 'Action', 'Romance', 'Crime', 'Mystery', 'TV Movie', 'Thriller', 'History', 'Documentary', 'Comedy', 'Family', 'Foreign'}


In [70]:
# Extract genre names
ohe_movies_df['genre_list'] = ohe_movies_df['genres'].apply(lambda x: [genre['name'] for genre in x] if isinstance(x, list) else [])
ohe_movies_df.head()

,genres,id,popularity,release_date,title,vote_average,vote_count,genre_list
26559,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",76600,6.020055,2020-12-16,Avatar 2,0.0,58.0,"[Action, Adventure, Fantasy, Science Fiction]"
38885,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",299782,0.238154,2018-12-31,The Other Side of the Wind,0.0,1.0,"[Comedy, Drama]"
30402,"[{'id': 53, 'name': 'Thriller'}, {'id': 28, 'n...",38700,2.178546,2018-11-07,Bad Boys for Life,0.0,12.0,"[Thriller, Action, Crime]"
38130,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...",332283,3.328261,2018-04-25,Mary Shelley,0.0,1.0,"[Drama, Romance]"
44535,"[{'id': 18, 'name': 'Drama'}]",412059,0.155147,2018-04-04,Mobile Homes,0.0,1.0,[Drama]


In [71]:
# One-hot encode genres

from sklearn.preprocessing import MultiLabelBinarizer

# Initialize MultiLabelBinarizer
mlb = MultiLabelBinarizer()

genre_ohe = pd.DataFrame(mlb.fit_transform(ohe_movies_df['genre_list']),
                         columns=mlb.classes_,
                         index=ohe_movies_df.index)

# Merge back into sample_genres_movies_df
ohe_movies_df = pd.concat([ohe_movies_df, genre_ohe], axis=1)

genre_list_mlb = mlb.classes_
genre_list = ohe_movies_df['genre_list']

# Drop unnecessary columns
ohe_movies_df = ohe_movies_df.drop(columns=['genres', 'genre_list'])

# print(sample_genres_movies_df.head())
ohe_movies_df

,id,popularity,release_date,title,vote_average,vote_count,Action,Adventure,Animation,Comedy,...,History,Horror,Music,Mystery,Romance,Science Fiction,TV Movie,Thriller,War,Western
26559,76600,6.020055,2020-12-16,Avatar 2,0.0,58.0,1,1,0,0,...,0,0,0,0,0,1,0,0,0,0
38885,299782,0.238154,2018-12-31,The Other Side of the Wind,0.0,1.0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
30402,38700,2.178546,2018-11-07,Bad Boys for Life,0.0,12.0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
38130,332283,3.328261,2018-04-25,Mary Shelley,0.0,1.0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
44535,412059,0.155147,2018-04-04,Mobile Homes,0.0,1.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45148,438910,0.001586,NaN,Engineering Red,6.0,2.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
45203,433711,0.00022,NaN,All Superheroes Must Die 2: The Last Superhero,4.0,1.0,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0
45338,335251,0.0,NaN,The Land Where the Blues Began,0.0,0.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
45410,449131,0.008903,NaN,Aprel,6.0,1.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [72]:
print(genre_list)
print("-----")
print(genre_list_mlb)

26559    [Action, Adventure, Fantasy, Science Fiction]
38885                                  [Comedy, Drama]
30402                        [Thriller, Action, Crime]
38130                                 [Drama, Romance]
44535                                          [Drama]
                             ...                      
45148                                               []
45203                       [Mystery, Science Fiction]
45338                                               []
45410                                   [Drama, Crime]
45461                                  [Drama, Family]
Name: genre_list, Length: 45465, dtype: object
-----
['Action' 'Adventure' 'Animation' 'Comedy' 'Crime' 'Documentary' 'Drama'
 'Family' 'Fantasy' 'Foreign' 'History' 'Horror' 'Music' 'Mystery'
 'Romance' 'Science Fiction' 'TV Movie' 'Thriller' 'War' 'Western']


In [73]:
genre_list

26559    [Action, Adventure, Fantasy, Science Fiction]
38885                                  [Comedy, Drama]
30402                        [Thriller, Action, Crime]
38130                                 [Drama, Romance]
44535                                          [Drama]
                             ...                      
45148                                               []
45203                       [Mystery, Science Fiction]
45338                                               []
45410                                   [Drama, Crime]
45461                                  [Drama, Family]
Name: genre_list, Length: 45465, dtype: object

In [96]:
ohe_movies_df.sort_values(by='id')

,id,popularity,release_date,title,vote_average,vote_count,Action,Adventure,Animation,Comedy,...,History,Horror,Music,Mystery,Romance,Science Fiction,TV Movie,Thriller,War,Western
4342,2,3.860491,1988-10-21,Ariel,7.1,44.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12947,3,2.29211,1986-10-16,Shadows in Paradise,7.1,35.0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
17,5,9.026586,1995-12-09,Four Rooms,6.5,539.0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
474,6,5.538671,1993-10-15,Judgment Night,6.4,79.0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
256,11,42.149697,1977-05-25,Star Wars,8.1,6778.0,1,1,0,0,...,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45078,465044,0.281008,2017-06-28,Abduction,0.0,0.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
45273,467731,0.001189,1956-02-19,Tragedy in a Temporary Town,0.0,0.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
21891,468343,0.001202,1956-01-01,Silja - nuorena nukkunut,0.0,0.0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
45398,468707,0.347806,2017-07-28,Thick Lashes of Lauri Mäntyvaara,8.0,1.0,0,0,0,1,...,0,0,0,0,1,0,0,0,0,0


In [97]:
# ratings_df.shape

In [98]:
# userId_ratings = ratings_df.groupby('userId').size().reset_index(name='count')
# userId_ratings = userId_ratings.sort_values(by='userId', ascending=False)
# userId_ratings.head()


# Creating a user profile

In [99]:
user_ratings = """
710,golden eye, 4
1770,Michael Collins, 5
76600,Avatar 2, 3
141052,Justice League, 2.5
284053,Thor: Ragnarok, 4
341013,Atomic Blonde, 3.8
374720,Dunkirk, 4.7
339403,Baby Driver, 4.7
324852,Despicable Me 3, 4
419192,McLaren, 3.5
283995,Guardians of the Galaxy Vol. 2, 3.8
324849,The Lego Batman Movie, 4.5
324552,John Wick: Chapter 2, 3.5
180863,T2 Trainspotting, 3.6
318846,The Big Short, 4.4
406,La Haine, 4.65
424,Schindler's List, 4.8
627,Trainspotting, 4.9
100,"Lock, Stock and Two Smoking Barrels", 4.3
107,Snatch, 4.9
1429,25th Hour, 3.3
49517,Tinker Tailor Soldier Spy, 4.1
6538,Charlie Wilson's War, 3.9
10315,Fantastic Mr. Fox, 4.7
27205,Inception, 3.6
106646,The Wolf of Wall Street, 4.5
120467,The Grand Budapest Hotel, 4.6
157336,Interstellar, 3.9
261023,Black Mass, 3.6
278,The Shawshank Redemption, 3
4232,Scream, 2.4
1893,Star Wars: Episode I - The Phantom Menace, 3.8
550,Fight Club, 4.2
98,Gladiator, 4
508,Love Actually, 2.1
228967,The Interview, 2.7
1895,Star Wars: Episode III - Revenge of the Sith, 3.5
18785,The Hangover, 1.9
67913,The Guard, 3.7
40807,50/50, 2.5
70160,The Hunger Games, 2.4
77930,Magic Mike, 1
72190,World War Z, 2.5
187017,22 Jump Street, 1.9
198663,The Maze Runner, 2
207703,Kingsman: The Secret Service, 2.1
99861,Avengers: Age of Ultron, 2
167073,Brooklyn, 2.8
254470,Pitch Perfect 2, 1.9
271718,Trainwreck, 1
314365,Spotlight, 4.9
259693,The Conjuring 2, 2.3
308266,War Dogs, 2.1
324786,Hacksaw Ridge, 3.1
330459,Rogue One: A Star Wars Story, 2.9
339846,Baywatch, 2
"""

In [100]:
from io import StringIO


# Convert into a DataFrame
user_ratings_df = pd.read_csv(StringIO(user_ratings), header=None, names=["movieId", "movie_name", "rating"])

# Drop the movie_name column as it's not needed for appending to the ratings DataFrame
user_ratings_df = user_ratings_df.drop(columns=["movie_name"])

user_ratings_df['movieId'] = pd.to_numeric(user_ratings_df['movieId'], errors='coerce')
user_ratings_df = user_ratings_df.dropna(subset=['movieId'])
user_ratings_df['movieId'] = user_ratings_df['movieId'].astype(int)

# Testing user id = 999999
user_ratings_df["userId"] = 999999

# Reorder columns to match the existing ratings DataFrame
user_ratings_df = user_ratings_df[["userId", "movieId", "rating"]]


# Print the DataFrame
print(user_ratings_df.head(3))


   userId  movieId  rating
0  999999      710     4.0
1  999999     1770     5.0
2  999999    76600     3.0


In [101]:
ratings_df = pd.concat([ratings_df, user_ratings_df], ignore_index=True)
# ratings_df = ratings_df.drop('timestamp', axis=1) 
ratings_df.head()
# ratings_df.tail(10)


,userId,movieId,rating,timestamp
0,1,31,2.5,1.260759e+09
1,1,1029,3.0,1.260759e+09
2,1,1061,3.0,1.260759e+09
3,1,1129,2.0,1.260759e+09
4,1,1172,4.0,1.260759e+09


In [102]:
# user_id_to_find = 999999
# entries_with_user_id = ratings_df[ratings_df['userId'] == user_id_to_find]
# entries_with_user_id


In [113]:

def create_user_profile(user_id, ratings_df, top_3_credits_df):
  """Creates a user profile based on ratings for movies with shared cast/directors."""

  user_ratings = ratings_df[ratings_df['userId'] == user_id]
  # print(user_ratings)
  profile = {}

  for _, rating_row in user_ratings.iterrows():
    movie_id = rating_row['movieId']
    rating = rating_row['rating']

    #find the index of the movie_id in top_3_credits_df
    try:
        movie_index = top_3_credits_df[top_3_credits_df['id'] == movie_id].index[0]
    except IndexError:
        continue # if the movie_id isn't in top_3_credits_df, skip this movie


    # Add cast and director IDs to user profile with weighted ratings
    cast_ids = top_3_credits_df['cast_info'].iloc[movie_index]
    for cast_member in cast_ids:
        profile[cast_member[1]] = profile.get(cast_member[1], 0) + rating

    director_info = top_3_credits_df['director_info'].iloc[movie_index]
    if director_info is not None:
      profile[director_info[1]] = profile.get(director_info[1], 0) + rating
    
    # Add genre IDs to user profile with weighted ratings
    genre_score = 0.1 * rating
    for genre in mlb.classes_:
        if ohe_movies_df[genre].iloc[movie_index] == 1:
            profile[genre] = profile.get(genre, 0) + genre_score
    

  return profile


# Example usage:
user_id = 999999  # Replace it with your desired user ID
user_profile = create_user_profile(user_id, ratings_df, top_3_credits_df)
# user_profile

In [114]:
save_ohe_movies_df = ohe_movies_df.copy()

In [115]:



# Convert the 'id' columns to numeric, forcing errors to NaN
ohe_movies_df.loc[:, 'id'] = pd.to_numeric(ohe_movies_df['id'], errors='coerce')
top_3_credits_df['id'] = pd.to_numeric(top_3_credits_df['id'], errors='coerce')

# Drop rows with NaN values in the 'id' columns
ohe_movies_df = ohe_movies_df.dropna(subset=['id'])
top_3_credits_df = top_3_credits_df.dropna(subset=['id'])

# Convert the 'id' columns to integers
ohe_movies_df.loc[:, 'id'] = ohe_movies_df['id'].astype(int)
top_3_credits_df['id'] = top_3_credits_df['id'].astype(int)

# Find common IDs
common_ids = set(ohe_movies_df['id']) & set(top_3_credits_df['id'])

print("Number of matching movie IDs:", len(common_ids))


Number of matching movie IDs: 45432


In [ ]:
def genre_weighting(genre_list, genre_list_mlb):
    """
    Assigns a weight to each genre based on the user's preferences.

    Args:
        genre_list (list): List of genres for a movie.
        genre_list_mlb (numpy.ndarray): Array of genre names.

    Returns:
        dict: Dictionary with genre names as keys and weights as values.
    """

    genre_weights = {}
    for genre in genre_list:
        genre_weights[genre] = 1

    return genre_weights

# Generate recommendations


In [116]:
user_profile

{517: 4.0,
 48: 4.0,
 10695: 4.0,
 '52fe426ec3a36847f801e14b': 4.0,
 'Comedy': 5.655000000000001,
 3896: 13.600000000000001,
 18992: 5.0,
 9029: 5.0,
 '52fe4313c3a36847f803877b': 5.0,
 'Horror': 2.86,
 'Mystery': 2.7,
 'Thriller': 3.950000000000001,
 204: 3.0,
 8691: 6.8,
 65731: 6.1,
 '52fe4943c3a368484e122b45': 3.0,
 'Crime': 1.78,
 'Romance': 2.9000000000000004,
 880: 2.5,
 73968: 2.5,
 90633: 2.5,
 '535e68b90e0a264fe10065f1': 2.5,
 'Drama': 7.440000000000002,
 74568: 6.0,
 91606: 4.0,
 112: 4.0,
 '562f1e69c3a3681b4d00e370': 4.0,
 6885: 3.8,
 5530: 3.8,
 568657: 3.8,
 '555da697925141757e0010d8': 3.8,
 'Music': 0.5900000000000001,
 1687041: 4.7,
 1765227: 4.7,
 1334638: 4.7,
 '56819cbe9251414f6300aab3': 4.7,
 1159982: 4.7,
 1016168: 4.7,
 1979: 4.7,
 '554d64de9251413e40000d7d': 4.7,
 'Adventure': 2.1,
 'War': 0.78,
 4495: 8.4,
 41091: 4.0,
 34517: 4.0,
 '57e71794c3a368222700add6': 4.0,
 'Western': 0.7100000000000001,
 218565: 3.5,
 90424: 3.5,
 1209543: 3.5,
 '57f1c03c9251414abf004c8

In [132]:
cast_ft_weight = 0.3
director_ft_weight = 0.3
genre_ft_weight = 0.15

def recommend_movies(user_id, user_profile, films, top_3_credits_df, ratings_df, top_n=20):
    """Recommends movies based on user profile and movie cast/director."""

    user_rated_movies = set(ratings_df[ratings_df['userId'] == user_id]['movieId'])
    unrated_movies = films[~films['id'].isin(user_rated_movies)]
    recommendations = []

    for _, movie_row in unrated_movies.iterrows():
      movie_id = movie_row['id']
      score, cast_score, director_score, genre_score = 0, 0, 0, 0
      
      #find the index of the movie_id in top_3_credits_df
      try:
          movie_index = top_3_credits_df[top_3_credits_df['id'] == movie_id].index[0]
      except IndexError:
          print(f"Movie ID {movie_id} not found in top_3_credits_df")
          continue
      
      # Calculate weighted score based on user profile and movie cast/director, genres, 
      cast_ids = top_3_credits_df['cast_info'].iloc[movie_index]
      for cast_member in cast_ids:
          cast_score += user_profile.get(cast_member[1], 0) * cast_ft_weight
      score += cast_score
    
      director_info = top_3_credits_df['director_info'].iloc[movie_index]
      if director_info is not None:
        director_score += user_profile.get(director_info[1], 0) * director_ft_weight
      score += director_score
      
      # Calculate weighted score based on one hot-encoded genres
      genre_score = sum([user_profile.get(genre, 0) for genre in mlb.classes_ if movie_row[genre] == 1])
      # if genre_score > 0:
      #     print(f"Movie ID: {movie_id}, Genre Score: {genre_score}")
      #     print(f"Movie ID: {movie_id}, Score: {score}")
      score += genre_score * genre_ft_weight
          
      # print(f"Movie ID: {movie_id}, Score: {score}")
      recommendations.append((movie_id, score, cast_score, director_score, genre_score))
    
    recommendations.sort(key=lambda x: x[1], reverse=True)  # Sort by score
    # print(recommendations)
    top_recommendations = recommendations[:top_n]
    
    movie_recs = [(movie_id, score, cast_score, director_score, genre_score) for movie_id, score, cast_score, director_score, genre_score in top_recommendations]
    # least_rated_movie_ids = [movie_id for movie_id, score in recommendations[:top_n]]
    
    # recommendations.sort(key=lambda x: x[1])  # Sort in ascending order
    lowest_recommendations = recommendations[-top_n:]
    least_movie_recs = [(movie_id, score, cast_score, director_score, genre_score) for movie_id, score, cast_score, director_score, genre_score in lowest_recommendations]
    
    return movie_recs, least_movie_recs

In [133]:
recommended_movies, least_rated_movie_ids = recommend_movies(user_id, user_profile, ohe_movies_df, top_3_credits_df, ratings_df)


Movie ID 401840 not found in top_3_credits_df


In [134]:
# Print recommended movies
print(recommended_movies)

ohe_movies_df[ohe_movies_df['id'].isin([movie_id for movie_id, _, _, _, _ in recommended_movies])][['title', 'release_date', 'vote_average', 'vote_count']].assign(
    score=[score for _, score, _, _, _ in recommended_movies], 
    cast_score=[cast_score for _, _, cast_score, _, _ in recommended_movies], 
    director_score=[director_score for _, _, _, director_score, _ in recommended_movies], 
    genre_score=[genre_score for _, _, _, _, genre_score in recommended_movies])


# ohe_movies_df[ohe_movies_df['id'].isin(recommended_movie_ids)][['title', 'release_date', 'vote_average', 'vote_count']]

[(174751, 9.6735, 8.01, 0.0, 11.090000000000003), (18823, 9.6105, 8.73, 0.0, 5.87), (57165, 9.045, 8.73, 0.0, 2.1), (51999, 9.0435, 7.29, 0.0, 11.690000000000003), (1894, 8.9385, 7.9799999999999995, 0.0, 6.390000000000001), (8066, 8.1735, 6.0600000000000005, 0.0, 14.090000000000003), (8067, 8.058750000000002, 4.74, 0.0, 22.125000000000007), (11324, 8.0535, 5.9399999999999995, 0.0, 14.090000000000003), (60308, 7.835999999999999, 6.719999999999999, 0.0, 7.440000000000002), (2212, 7.7235, 4.74, 0.0, 19.89), (1808, 7.264500000000001, 6.0600000000000005, 0.0, 8.030000000000003), (272, 7.224, 5.4, 0.0, 12.160000000000002), (43566, 7.198500000000001, 5.49, 0.0, 11.390000000000004), (87492, 7.168500000000001, 5.46, 0.0, 11.390000000000004), (294652, 7.156500000000001, 4.74, 0.0, 16.110000000000003), (237, 7.150500000000001, 4.74, 0.0, 16.070000000000004), (55347, 7.1392500000000005, 4.74, 0.0, 15.995000000000003), (8952, 7.1392500000000005, 4.74, 0.0, 15.995000000000003), (9450, 7.139250000000

,title,release_date,vote_average,vote_count,score,cast_score,director_score,genre_score
35696,Jane Got a Gun,2016-01-01,5.4,293.0,9.67350,8.01,0.0,11.090
25857,Son of a Gun,2014-10-16,6.1,284.0,9.61050,8.73,0.0,5.870
24640,Foxcatcher,2014-05-19,6.5,965.0,9.04500,8.73,0.0,2.100
18779,Wrath of the Titans,2012-03-27,5.5,1459.0,9.04350,7.29,0.0,11.690
17758,Moneyball,2011-09-22,7.0,1409.0,8.93850,7.98,0.0,6.390
18559,Perfect Sense,2011-01-24,6.9,298.0,8.17350,6.06,0.0,14.090
17237,Beginners,2010-09-11,6.8,353.0,8.05875,4.74,0.0,22.125
15006,Clash of the Titans,2010-04-01,5.6,2280.0,8.05350,5.94,0.0,14.090
14825,Shutter Island,2010-02-18,7.8,6559.0,7.83600,6.72,0.0,7.440
14141,The Men Who Stare at Goats,2009-10-17,5.9,755.0,7.72350,4.74,0.0,19.890


In [ ]:
# Sausage Party
# Miles Ahead
# Jane Got a Gun
# Mortdecai
# Foxcatcher
# Wrath of the Titans
# Moneyball
# Perfect Sense
# Clash of the Titans
# Shutter Island
# The Ghost Writer
# The Men Who Stare at Goats
# Angels & Demons
# Seraphim Falls
# Stay
# Batman Begins
# Star Wars: Episode II - Attack of the Clones
# Velvet Goldmine
# Before and After
# Rob Roy


In [86]:
ohe_movies_df[ohe_movies_df['id'].isin(least_rated_movie_ids)][['title', 'release_date', 'vote_average', 'vote_count']]


,title,release_date,vote_average,vote_count
42179,Patient Zero,NaN,10.0,2.0
42568,Winning Favour,NaN,0.0,0.0
42573,Whn the day had no name,NaN,7.0,2.0
42941,Blindpassasjer,NaN,7.0,1.0
43090,Half -Life,NaN,3.7,3.0
43377,Jedi Junior High,NaN,10.0,1.0
43523,Cosmos,NaN,9.1,41.0
43962,Irwin & Fran 2013,NaN,0.0,0.0
44014,Shivering Trunks,NaN,0.0,0.0
44065,Supermassive Black Holes,NaN,7.0,1.0


Next, let's look at the ratings dataframe.


In [41]:
#Drop removes a specified row or column from a dataframe
ratings_df = ratings_df.drop('timestamp', axis=1)
ratings_df.head()

,userId,movieId,rating
0,1,31,2.5
1,1,1029,3.0
2,1,1061,3.0
3,1,1129,2.0
4,1,1172,4.0
